# 06 — DSPy: Optimized Complaint Triage Module

**Purpose:** given a new complaint and the similar historical cases retrieved in Notebook 05, generate a structured triage summary — a likely issue category, a suggested resolution approach, and a confidence level — using DSPy to optimize the prompt automatically against a labeled example set, rather than hand-tuning it by trial and error. This is the final stage of the pipeline: Notebook 04 established which signals predict outcomes, Notebook 05 built the retrieval layer, and this notebook turns both into a single triage tool a caseworker could actually use on a new complaint.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/ANLP_portfolio/")


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


## Model Setup

Using Gemini 3.1 Flash-Lite via API, after two other options didn't pan out for this environment:

**Ollama was the original plan**, since a local model avoids per-call costs across the many LLM calls this notebook makes (the sanity check, the optimizer's bootstrapping pass, and the baseline-vs-optimized comparison). But Ollama needs a local server running on the same machine — it has no path to running inside Colab's hosted runtime, so it was never viable here once the notebook moved off a local environment.

**Gemini 2.5 (Flash and Flash-Lite) was the next option**, but new API keys are currently blocked from the entire 2.5 generation — Google returns a 404 ("no longer available to new users") for 2.5 Pro, 2.5 Flash, and 2.5 Flash-Lite alike on recently created keys, routing new users to the 3.x generation instead. Older keys can still reach 2.5; a fresh key created for this project can't.

**Gemini 3.1 Flash-Lite** is the resolution — a small, fast, cost-efficient model in the generation new keys are actually granted access to, and a reasonable fit for a classification/routing task like this one that doesn't need heavy reasoning. Worth keeping an eye on rate limits either way: this notebook's call volume is real (bootstrapping + evaluation), so the reduced 15/10 train/eval example counts below are sized to stay comfortably inside whatever free-tier quota applies.

In [2]:
!pip install -q -U dspy-ai google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 38.7 MB/s eta 0:00:00


In [2]:
import dspy
print(dspy.__version__)

3.2.1


In [3]:
import os
from google.colab import userdata
import dspy

api_key = userdata.get("GEMINI_API_KEY")

lm = dspy.LM(
    "gemini/gemini-3.1-flash-lite",
    api_key=api_key
)

dspy.configure(lm=lm)

In [4]:
response = lm("Summarize: Customer disputed a credit card charge after fraud.")
print(response)

['Here are a few ways to summarize that statement, depending on your needs:\n\n*   **Concise:** The customer filed a fraud-related chargeback.\n*   **Formal:** The customer has initiated a dispute due to unauthorized credit card activity.\n*   **Action-oriented:** The customer is contesting a charge, citing fraud.\n*   **Brief:** Fraud-based chargeback filed.']


## Define the Signature

The signature specifies inputs (the new complaint + retrieved similar cases) and outputs (a structured triage summary) — DSPy uses this to know what it's optimizing prompts toward.

In [5]:
class ComplaintTriage(dspy.Signature):
    """Given a new consumer complaint and similar historical complaints with their
    resolutions, produce a concise triage summary: likely category, suggested
    resolution approach, and confidence level."""

    new_complaint: str = dspy.InputField(desc="The new complaint narrative")
    similar_cases: str = dspy.InputField(desc="Similar historical complaints and how they were resolved")

    predicted_category: str = dspy.OutputField(desc="Best-guess issue category")
    suggested_approach: str = dspy.OutputField(desc="Recommended resolution approach based on similar cases")
    confidence: str = dspy.OutputField(desc="High / Medium / Low, based on how closely similar_cases match")

## Build the Module

In [6]:
class TriageModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.triage = dspy.ChainOfThought(ComplaintTriage)

    def forward(self, new_complaint, similar_cases):
        return self.triage(new_complaint=new_complaint, similar_cases=similar_cases)

triage_module = TriageModule()

## Wire In the Retriever from Notebook 05

Rebuilding the retriever using the exact same corpus, train/test split, and embedding model as Notebook 05, so what's indexed here is identical to what was validated there (0.86 mean issue-match rate on held-out queries). Persisting the FAISS index to disk on first build (`FAISS.save_local`) and loading it back on subsequent runs, so this notebook doesn't re-embed 131,564 documents every time it's opened.

In [11]:
!pip install -q langchain-community langchain-text-splitters faiss-cpu sentence-transformers --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

INDEX_PATH = "data/05_faiss_index"

df = pd.read_csv("data/04_complaints_model_ready.csv", low_memory=False)
df = df.dropna(subset=['Consumer complaint narrative'])

TARGET_MERGE_MAP = {
    'In progress': 'Other',
    'Untimely response': 'Other',
}
y_merged = df['Company response to consumer'].replace(TARGET_MERGE_MAP)

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y_merged
)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(INDEX_PATH):
    print("Loading persisted FAISS index from Notebook 05...")
    vectorstore = FAISS.load_local(INDEX_PATH, embedding_model, allow_dangerous_deserialization=True)
else:
    print("No persisted index found. Rebuilding from the training partition (same as Notebook 05)...")
    corpus_df = df.iloc[train_idx]
    documents = [
        Document(
            page_content=row['Consumer complaint narrative'],
            metadata={
                'product': row.get('Product'),
                'issue': row.get('Issue'),
                'topic': row.get('topic_name'),
                'has_dollar_amount': row.get('has_dollar_amount'),
                'has_company_org': row.get('has_company_org'),
                'company': row.get('Company'),
                'response': row.get('Company response to consumer'),
                'complaint_id': row.get('Complaint ID'),
            }
        )
        for _, row in corpus_df.iterrows()
    ]
    vectorstore = FAISS.from_documents(documents, embedding_model)
    vectorstore.save_local(INDEX_PATH)
    print(f"Built and saved index to {INDEX_PATH} for reuse.")

retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
print(f"Retriever ready. Held-out test partition size: {len(test_idx)}")

/tmp/ipykernel_9682/1535660792.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_9682/1535660792.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

No persisted index found. Rebuilding from the training partition (same as Notebook 05)...
Built and saved index to data/05_faiss_index for reuse.
Retriever ready. Held-out test partition size: 32891


In [8]:
def format_similar_cases(retrieved_docs):
    return "\n\n".join(
        f"Complaint: {d.page_content[:300]}\nResolution: {d.metadata['response']}"
        for d in retrieved_docs
    )

test_row = df.iloc[test_idx].iloc[0]
retrieved = retriever.invoke(test_row['Consumer complaint narrative'])
similar_cases_text = format_similar_cases(retrieved)

result = triage_module(new_complaint=test_row['Consumer complaint narrative'], similar_cases=similar_cases_text)
print(f"True issue: {test_row['Issue']}\n")
print(f"Predicted category: {result.predicted_category}")
print(f"Suggested approach: {result.suggested_approach}")
print(f"Confidence: {result.confidence}")

True issue: Incorrect information on your report

Predicted category: Credit Reporting - Inaccurate Information / Identity Theft
Suggested approach: Initiate a formal FCRA dispute investigation for each of the listed accounts. Cross-reference the account data provided by the consumer against the information furnished by the data providers. If documentation of identity theft is present, proceed with account blocking; otherwise, provide a detailed response confirming whether the items have been verified, updated, or deleted.
Confidence: High


### Observation: Sanity Check Prediction

The one-example sanity check came back substantively correct — I got the right underlying issue (the model called it "Credit Reporting - Inaccurate Information / Identity Theft" against the true "Incorrect information on your report") plus a specific, procedurally sound FCRA-based resolution approach at high confidence.

This also caught a real limitation in my evaluation metric before I ran the full eval set: the keyword-overlap check would score this prediction as a miss, since "inaccurate" and "incorrect" don't share a literal substring even though they mean the same thing here. I'm keeping the metric as-is rather than tuning it to catch this case, but flagging that my eventual accuracy number is a lower bound on the module's real performance, not a precise measure of it — individual "failures" are worth a manual glance before I take them at face value.

## Build an Evaluation Set and Optimize

Building a labeled example set from the held-out test partition (`test_idx`) — the same partition used for the retrieval quality check in Notebook 05, so nothing here has ever been indexed into the retriever's corpus. Each example pairs a complaint and its retrieved similar cases with the complaint's true `Issue` label.

Splitting the held-out partition into two disjoint slices: one for the optimizer to bootstrap few-shot demonstrations from, and a separate one reserved for the final baseline-vs-optimized comparison — so the optimizer is never evaluated on examples it had a chance to learn from. Using 15 training examples and 10 eval examples (smaller than the original 40/30 split) to stay well inside the Flash-Lite free-tier rate limit.

**Metric design note:** `predicted_category` is free-generated text, not a value pulled from a fixed label set, so an exact string match against the true `Issue` label would fail even when the prediction is substantively correct (e.g., predicting "incorrect account information" against the true label "Incorrect information on your report"). Using a keyword-overlap check instead — the metric passes if a meaningful word from the true `Issue` label appears in the predicted category — a looser but more honest measure of whether the module is getting the right idea, not just the exact string.

In [9]:
import re
import random

def build_examples(df, indices, retriever, n, seed=42):
    rng = random.Random(seed)
    sampled = rng.sample(list(indices), min(n, len(indices)))
    examples = []
    for idx in sampled:
        row = df.iloc[idx]
        retrieved = retriever.invoke(row['Consumer complaint narrative'])
        similar_cases_text = format_similar_cases(retrieved)
        ex = dspy.Example(
            new_complaint=row['Consumer complaint narrative'],
            similar_cases=similar_cases_text,
            predicted_category=row['Issue'],
        ).with_inputs('new_complaint', 'similar_cases')
        examples.append(ex)
    return examples

# Disjoint train/eval slices from the held-out test partition.
test_idx_list = list(test_idx)
rng = random.Random(42)
rng.shuffle(test_idx_list)
optimizer_train_idx = test_idx_list[:15]
eval_idx = test_idx_list[15:25]

train_examples = build_examples(df, optimizer_train_idx, retriever, n=15)
eval_examples = build_examples(df, eval_idx, retriever, n=10)

def triage_metric(example, prediction, trace=None):
    true_label = example.predicted_category.lower()
    pred = prediction.predicted_category.lower()
    keywords = [w for w in re.findall(r'\w+', true_label) if len(w) > 4]
    return any(kw in pred for kw in keywords)

print(f"Train examples (for optimizer): {len(train_examples)}")
print(f"Eval examples (held out from optimizer): {len(eval_examples)}")

Train examples (for optimizer): 15
Eval examples (held out from optimizer): 10


### Checking Whether the Optimizer's Metric Actually Filters

Before trusting what `BootstrapFewShot` selects as "successful" demonstrations, checking what fraction of the 15 training examples the baseline module already passes under `triage_metric` — the same metric later shown to pass on the generic word "report" alone. If most training examples pass regardless of correctness, the optimizer isn't really filtering for good demonstrations, just for ones that happen to share vocabulary with their true label — and `optimized_triage`'s few-shot examples should be read with that in mind, not treated as curated "correct" cases.

In [29]:
train_pass_count = sum(
    triage_metric(ex, triage_module(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases))
    for ex in train_examples
)
print(f"Training examples passing triage_metric (pre-optimization): {train_pass_count}/{len(train_examples)}")

Training examples passing triage_metric (pre-optimization): 13/15


In [13]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(metric=triage_metric, max_bootstrapped_demos=4, max_labeled_demos=8)
optimized_triage = optimizer.compile(triage_module, trainset=train_examples)

 33%|███▎      | 5/15 [00:00<00:00, 22.19it/s]

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.


### Caveat: What the Optimizer Actually Selected

**Result: 13 of 15 (87%) training examples passed `triage_metric` before optimization.** This confirms the concern — the metric barely filtered anything. `optimized_triage`'s 4 bootstrapped demonstrations were drawn from a pool where correctness and vocabulary overlap were nearly indistinguishable, so the optimizer wasn't curating genuinely correct examples so much as examples that happened to use overlapping words. This context matters for how the later semantic-metric improvement (0.12 → 0.36) should be read — see the failure-pattern check further down for what this weak filtering actually produced.

## Compare Optimized vs. Unoptimized

Before accepting the optimizer's reported improvement, first verifying that the optimization metric is actually discriminative.

Running both the baseline `triage_module` (zero-shot chain-of-thought, no examples) and `optimized_triage` (bootstrapped few-shot demonstrations from the optimizer) over the same held-out `eval_examples`, using the same keyword-overlap metric. This is the concrete, quantified payoff of using DSPy's optimizer instead of a hand-written prompt — worth reporting honestly even if the improvement turns out to be modest, consistent with how every prior stage of this project reported its results.

In [14]:
def evaluate(module, examples, metric):
    correct = 0
    for ex in examples:
        pred = module(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
        if metric(ex, pred):
            correct += 1
    return correct / len(examples)

baseline_acc = evaluate(triage_module, eval_examples, triage_metric)
optimized_acc = evaluate(optimized_triage, eval_examples, triage_metric)

print(f"Baseline (unoptimized) accuracy:  {baseline_acc:.2f}")
print(f"Optimized accuracy:               {optimized_acc:.2f}")
print(f"Absolute improvement:             {optimized_acc - baseline_acc:+.2f}")

Baseline (unoptimized) accuracy:  0.90
Optimized accuracy:               1.00
Absolute improvement:             +0.10


### Inspecting the One Flipped Example

The aggregate numbers (baseline 0.90, optimized 1.00) come from a 10-example eval set, so the entire "+0.10 improvement" is a single example flipping from wrong to right — not enough to call this a systematic effect of the optimizer's bootstrapped demonstrations versus a sampling artifact. Combined with the bootstrapping step's own signal (4 of the first 5 training examples already passed the metric on the unoptimized module), there wasn't much headroom for the optimizer to improve on to begin with.

Pulling the specific example that changed, to see whether it's a genuine correction or another synonym-matching artifact like the one caught in the earlier sanity check, before deciding what this result actually supports in the writeup.

In [15]:
for ex in eval_examples:
    base_pred = triage_module(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
    opt_pred = optimized_triage(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
    base_ok = triage_metric(ex, base_pred)
    opt_ok = triage_metric(ex, opt_pred)
    if base_ok != opt_ok:
        print(f"True: {ex.predicted_category}")
        print(f"Baseline: {base_pred.predicted_category} -> {'PASS' if base_ok else 'FAIL'}")
        print(f"Optimized: {opt_pred.predicted_category} -> {'PASS' if opt_ok else 'FAIL'}")
        print()

True: Incorrect information on your report
Baseline: Identity Theft / Fraud Dispute -> FAIL
Optimized: Improper use of your report -> PASS



### Checking Whether the Metric Can Actually Discriminate

The flip wasn't a real correction — the optimized model's "pass" came from sharing the word "report" with the true label, not from naming the same issue. Since this dataset is scoped to credit reporting, several `Issue` categories likely end in "...your report," which would make the keyword-overlap metric reward any report-related answer regardless of specificity. Checking how widespread that is.

In [16]:
issue_labels = df['Issue'].unique()
report_containing = [i for i in issue_labels if 'report' in i.lower()]
print(f"{len(report_containing)} of {len(issue_labels)} Issue labels contain 'report':")
for i in report_containing:
    print(f"  {i}")

3 of 6 Issue labels contain 'report':
  Incorrect information on your report
  Improper use of your report
  Unable to get your credit report or credit score


### Confirming the Word "Report" Is Doing the Work

If predictions frequently contain "report" independent of correctness, the metric is passing on vocabulary, not category accuracy.

In [17]:
report_count_baseline = sum('report' in triage_module(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases).predicted_category.lower() for ex in eval_examples)
report_count_optimized = sum('report' in optimized_triage(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases).predicted_category.lower() for ex in eval_examples)

print(f"Baseline predictions containing 'report': {report_count_baseline}/{len(eval_examples)}")
print(f"Optimized predictions containing 'report': {report_count_optimized}/{len(eval_examples)}")

Baseline predictions containing 'report': 9/10
Optimized predictions containing 'report': 10/10


## Diagnostic: The Accuracy Numbers Are Inflated by a Generic Word

The flipped example between baseline and optimized turned out not to be a genuine correction — investigating it surfaced a structural flaw in the evaluation metric itself.

Three of the six `Issue` labels in this credit-reporting-scoped dataset contain the word "report," and they're not evenly weighted — "Incorrect information on your report" and "Improper use of your report" are among the largest classes in the dataset. Checking the model's predictions directly confirmed the exposure: 9 of 10 baseline predictions and 10 of 10 optimized predictions contain "report," almost regardless of whether the prediction names the correct issue.

Since the keyword-overlap metric passes on any shared word longer than 4 characters, "report" alone is enough to pass against any of the three report-containing true labels — which cover a majority of this eval set. The reported 0.90 baseline / 1.00 optimized accuracy is inflated by this generic domain word, not a trustworthy measure of whether either module identifies the correct issue.

**Fix:** exclude domain-generic terms ("report," "your," "credit," "information") from the keyword set before checking overlap, so the metric only credits matches on words that actually distinguish one `Issue` category from another.

In [18]:
# Domain-generic terms that appear across most Issue categories in this
# credit-reporting-scoped dataset and therefore don't distinguish between
# them -- "report" is the clearest offender (found in 3 of 6 labels,
# covering the majority of the eval set), but excluding the full set of
# words that appear in nearly every label, not just that one.
GENERIC_TERMS = {
    'your', 'report', 'credit', 'information', 'with', 'from', 'about',
    'company', 'account', 'reporting',
}

def triage_metric_strict(example, prediction, trace=None):
    true_label = example.predicted_category.lower()
    pred = prediction.predicted_category.lower()
    keywords = [
        w for w in re.findall(r'\w+', true_label)
        if len(w) > 4 and w not in GENERIC_TERMS
    ]
    if not keywords:
        # Fallback: if every word in the true label is generic (shouldn't
        # happen given the label set, but guarding against it), fall back
        # to the original less-strict check rather than auto-failing.
        keywords = [w for w in re.findall(r'\w+', true_label) if len(w) > 4]
    return any(kw in pred for kw in keywords)

In [19]:
baseline_acc_strict = evaluate(triage_module, eval_examples, triage_metric_strict)
optimized_acc_strict = evaluate(optimized_triage, eval_examples, triage_metric_strict)

print(f"Baseline (unoptimized) accuracy, strict metric:  {baseline_acc_strict:.2f}")
print(f"Optimized accuracy, strict metric:                {optimized_acc_strict:.2f}")
print(f"Absolute improvement:                             {optimized_acc_strict - baseline_acc_strict:+.2f}")
print()
print(f"For comparison, original (generic-word-inflated) numbers were:")
print(f"Baseline: {baseline_acc:.2f}, Optimized: {optimized_acc:.2f}")

Baseline (unoptimized) accuracy, strict metric:  0.00
Optimized accuracy, strict metric:                0.40
Absolute improvement:                             +0.40

For comparison, original (generic-word-inflated) numbers were:
Baseline: 0.90, Optimized: 1.00


### Checking Whether the Strict Metric Over-Corrected

Neither 0.00 nor 0.40 is trustworthy without inspection — the swing from the generic-word-inflated numbers is too large to accept at face value. Printing the true label alongside both modules' predictions for every eval example, to see whether the baseline's failures are genuine category misses or synonym mismatches (paraphrasing instead of matching CFPB's exact taxonomy wording) like the one already caught in the sanity check. If exact-word exclusion turns out to over-correct, the next step is a category-aware semantic metric instead of just removing more words.

In [20]:
for ex in eval_examples:
    base_pred = triage_module(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
    opt_pred = optimized_triage(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
    print(f"True:      {ex.predicted_category}")
    print(f"Baseline:  {base_pred.predicted_category}")
    print(f"Optimized: {opt_pred.predicted_category}")
    print()

True:      Incorrect information on your report
Baseline:  Credit Reporting / Accuracy / Obsolete Information
Optimized: Incorrect information on your report

True:      Incorrect information on your report
Baseline:  Credit Reporting - Identity Theft / Fraud
Optimized: Identity theft / Fraudulent reporting

True:      Incorrect information on your report
Baseline:  Credit reporting / Debt collection / Lending practices
Optimized: Incorrect information on your report

True:      Incorrect information on your report
Baseline:  Identity Theft / Fraudulent Reporting
Optimized: Identity theft / Fraudulent reporting

True:      Incorrect information on your report
Baseline:  Identity Theft / Fraudulent Reporting
Optimized: Improper use of your report

True:      Incorrect information on your report
Baseline:  Credit Reporting / Investigation of Inaccurate Information
Optimized: Incorrect information on your report

True:      Incorrect information on your report
Baseline:  Identity Theft / 

## Diagnostic: The Strict Comparison Reveals Vocabulary Imitation, Not Clean Improvement

Manually inspecting all 10 eval predictions clarified what the strict-metric swing (0.00 baseline, 0.40 optimized) actually represents. The optimized module adopted CFPB's exact taxonomy wording — 8 of 10 predictions are one of three literal label strings, almost certainly copied from the optimizer's bootstrapped demonstrations. The baseline never once used exact CFPB wording, consistently paraphrasing instead.

That's a real behavioral change from few-shot optimization, but not a clean accuracy win. The optimized module confuses the two majority classes in both directions — predicting "Improper use of your report" for true "Incorrect information" twice, and the reverse twice — consistent with pattern-matching to a small set of memorized demo labels rather than reasoning from each complaint's actual content. With only 4 bootstrapped demonstrations, some correct predictions may be frequency luck among 2-3 memorized strings rather than genuine discrimination.

The baseline's 0.00 is also not fully trustworthy as "wrong every time" — at least one baseline prediction ("Unauthorized Inquiries" for true "Improper use of your report") is a defensible synonym that fails the strict metric on wording alone, not on substance.

**Honest read:** optimization changed the module's output style — toward CFPB's own vocabulary — more clearly than it changed its underlying accuracy. Distinguishing genuine improvement from vocabulary imitation would need a larger eval set and likely a metric that credits semantic equivalence, not just literal keyword overlap.

### Full Category List, for a Category-Aware Metric

Getting all six `Issue` categories so the next metric can be built with signature keywords for each one, not just the three already seen.

In [21]:
# issue_labels already computed above; just printing the full list here.
print(f"{len(issue_labels)} total Issue categories:")
for i in issue_labels:
    print(f"  {i}")

6 total Issue categories:
  Incorrect information on your report
  Problem with a company's investigation into an existing problem
  Improper use of your report
  Unable to get your credit report or credit score
  Problem with fraud alerts or security freezes
  Credit monitoring or identity theft protection services


### Building a Semantic-Equivalence Metric for All Six Categories

Mapping each of the six `Issue` categories to a small set of distinguishing keywords/phrases — chosen to be substantively meaningful to that category and, as much as possible, not shared across categories (unlike "report," which was the problem with the original metric). This is a heuristic proxy for semantic equivalence, not a true semantic check — it can still miscount a genuinely correct prediction that phrases things unusually, or wrongly credit a prediction that happens to use one of these words out of context. Worth stating that limitation directly in the writeup rather than treating this as a solved problem, the same way the keyword-overlap version's limitations were stated once found.

Also scaling up the eval set (from 10 to 25) so "accuracy" reflects more than a handful of examples, while keeping the optimizer's training set at 15 (unchanged, so `optimized_triage` doesn't need to be recompiled) — reusing the same disjoint-partition logic so the larger eval slice still never overlaps with the training slice.

In [25]:
# Keyword sets per category -- chosen to distinguish between categories,
# not just describe them. Deliberately excludes generic terms already
# shown to be uninformative here ("report," "credit," "information").
CATEGORY_KEYWORDS = {
    "Incorrect information on your report": [
        "incorrect", "inaccurate", "wrong", "erroneous", "false"
    ],
    "Problem with a company's investigation into an existing problem": [
        "investigation", "reinvestigat", "dispute process", "dispute handling", "mishandled"
    ],
    "Improper use of your report": [
        "improper", "unauthorized", "misuse", "inquiry", "inquiries", "impermissible"
    ],
    "Unable to get your credit report or credit score": [
        "unable to", "can't get", "cannot get", "denied access", "access to", "obtain"
    ],
    "Problem with fraud alerts or security freezes": [
        "fraud alert", "security freeze", "freeze", "lock"
    ],
    "Credit monitoring or identity theft protection services": [
        "monitoring", "protection service", "subscription", "identity theft protection"
    ],
}

def triage_metric_semantic(example, prediction, trace=None):
    true_label = example.predicted_category
    pred = prediction.predicted_category.lower()
    keywords = CATEGORY_KEYWORDS.get(true_label, [])
    return any(kw in pred for kw in keywords)

In [26]:
# Scaling eval to 25, keeping the same 15-example training slice (no
# optimizer recompile needed) and the same disjoint-partition logic.
eval_idx_large = test_idx_list[15:40]
eval_examples_large = build_examples(df, eval_idx_large, retriever, n=25)
print(f"Eval examples (enlarged): {len(eval_examples_large)}")

Eval examples (enlarged): 25


In [27]:
baseline_acc_semantic = evaluate(triage_module, eval_examples_large, triage_metric_semantic)
optimized_acc_semantic = evaluate(optimized_triage, eval_examples_large, triage_metric_semantic)

print(f"Baseline (unoptimized) accuracy, semantic metric:  {baseline_acc_semantic:.2f}")
print(f"Optimized accuracy, semantic metric:                {optimized_acc_semantic:.2f}")
print(f"Absolute improvement:                               {optimized_acc_semantic - baseline_acc_semantic:+.2f}")

Baseline (unoptimized) accuracy, semantic metric:  0.12
Optimized accuracy, semantic metric:                0.36
Absolute improvement:                               +0.24


### Checking Whether Low Absolute Accuracy Reflects the Model or the Metric

Both accuracy numbers are lower than expected given how substantively reasonable the sanity-check prediction looked earlier. Before concluding the module performs poorly at this task, checking whether some of the failures are keyword-list gaps — e.g., "obsolete" or "accuracy" aren't in the current keyword set for "Incorrect information on your report," even though a prediction using those words could be substantively correct. Printing baseline failures specifically, since that's the larger failure count (22 of 25), to see how many look like genuine misses versus keyword coverage gaps.

In [28]:
for ex in eval_examples_large:
    base_pred = triage_module(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
    if not triage_metric_semantic(ex, base_pred):
        print(f"True: {ex.predicted_category}")
        print(f"Baseline: {base_pred.predicted_category}")
        print()

True: Improper use of your report
Baseline: Debt Collection / Validation Request

True: Improper use of your report
Baseline: Credit reporting / Credit repair services

True: Incorrect information on your report
Baseline: Credit Reporting - Identity Theft / Fraud

True: Incorrect information on your report
Baseline: Identity Theft / Fraud Dispute

True: Incorrect information on your report
Baseline: Credit Reporting / Identity Theft

True: Incorrect information on your report
Baseline: Credit Reporting Error - Student Loan Deferment

True: Incorrect information on your report
Baseline: Credit reporting / Debt collection / Lending practices

True: Problem with a company's investigation into an existing problem
Baseline: Credit Reporting / Identity Theft / Debt Validation

True: Incorrect information on your report
Baseline: Identity Theft / Fraudulent Reporting

True: Incorrect information on your report
Baseline: Credit Reporting / Identity Theft / Data Breach

True: Incorrect informat

### Diagnostic: Baseline Failures Are Mostly Genuine Confusions, Not Keyword Gaps

Manually reviewing all 22 baseline failures: only 1–2 look like keyword-list coverage gaps (a prediction like "Accuracy / Obsolete Information" plausibly means the same thing as "incorrect information" but shares no listed keyword). The remaining ~20 are genuine category confusions — the baseline predicted a substantively different issue, not just different wording for the right one.

The dominant failure pattern: 10 of 22 failures (45%) predict "Identity Theft" or "Fraud" as the category, despite none of the true labels in this failure set being identity-theft-related. This looks like a systematic bias rather than random noise — plausibly because narratives mentioning fraud or identity theft appear often among the retrieved similar cases, pulling the baseline's free-text categorization toward that framing even when the true `Issue` is more mundane. A secondary pattern repeats the cross-confusion between "Incorrect information on your report" and "Improper use of your report" already seen in the optimized module's errors on the smaller eval set.

This means the +0.24 gap between baseline (0.12) and optimized (0.36) reflects a mostly genuine difference in model behavior, not primarily a metric artifact — the baseline has a real tendency to over-predict identity-theft framing that the optimized module, primed with CFPB's own label vocabulary via its bootstrapped demonstrations, doesn't share as strongly.

### Checking the Optimized Module's Failures at the Same Scale

The claim that the optimized module "doesn't share the baseline's identity-theft bias as strongly" was only verified on the earlier 10-example strict-metric set. Running the same check on `eval_examples_large` so it's backed by the same data as the headline numbers.

In [30]:
for ex in eval_examples_large:
    opt_pred = optimized_triage(new_complaint=ex.new_complaint, similar_cases=ex.similar_cases)
    if not triage_metric_semantic(ex, opt_pred):
        print(f"True: {ex.predicted_category}")
        print(f"Optimized: {opt_pred.predicted_category}")
        print()

True: Improper use of your report
Optimized: Debt collection / Validation of debt

True: Improper use of your report
Optimized: Incorrect information on your report

True: Incorrect information on your report
Optimized: Identity theft / Fraudulent reporting

True: Incorrect information on your report
Optimized: Improper use of your report

True: Problem with a company's investigation into an existing problem
Optimized: Incorrect information on your report

True: Improper use of your report
Optimized: Incorrect information on your report

True: Incorrect information on your report
Optimized: Improper use of your report

True: Incorrect information on your report
Optimized: Identity theft / Fraudulent reporting

True: Incorrect information on your report
Optimized: Improper use of your report

True: Improper use of your report
Optimized: Problem with a credit reporting company's investigation into an existing problem

True: Improper use of your report
Optimized: Incorrect information on 

### Diagnostic: The Optimized Module Traded One Bias for Another

Ran the same failure check on `optimized_triage` against `eval_examples_large`. 17 failures printed (slightly more than the 16 implied by the 0.36 accuracy figure — a 1-example discrepancy consistent with the LLM's non-deterministic output across separate calls, not a bug; these numbers aren't perfectly reproducible run to run without caching or a fixed seed).

Of the 17 failures: **9 (~53%) are cross-confusions between "Incorrect information on your report" and "Improper use of your report"** — the two largest categories — in both directions. Only 4 (~24%) are identity-theft/fraud mispredictions, down from the baseline's 45%, so the earlier bias-reduction claim holds directionally but is less clean than it first looked.

Combined with the 87% pre-optimization pass rate above, this reframes what optimization actually did here: rather than teaching the module genuine discrimination between categories, weak demo filtering left it leaning on CFPB's exact label vocabulary (already established) and defaulting toward the two most frequent categories when uncertain — trading the baseline's identity-theft over-prediction for a narrower, majority-class flip-flop instead. Net accuracy improved, but the improvement looks more like a shift in which bias dominates than acquired reasoning.

## Summary and Takeaway

This notebook's real finding ended up being as much about evaluation methodology as about the triage module itself, and I think that's worth stating plainly rather than glossing over.

My first metric — keyword overlap on any word over 4 characters — looked fine on the surface (baseline 0.90, optimized 1.00) but turned out to be nearly meaningless: half of this dataset's six `Issue` categories contain the word "report," and 9 of 10 baseline predictions and all 10 optimized predictions happened to contain that word too, regardless of whether they named the correct issue. A metric that passes on a domain-generic word isn't measuring what I need it to measure.

Tightening the metric to exclude generic terms swung the numbers hard the other way (0.00 baseline, 0.40 optimized on the same 10 examples), and I didn't trust that swing either — a jump that large usually means I overcorrected, not that I found the truth. Inspecting the actual predictions confirmed a real, interesting behavior: the optimized module had adopted CFPB's exact label vocabulary from its bootstrapped demonstrations, while the baseline consistently paraphrased into its own words. That's a genuine effect of few-shot optimization, but it's a vocabulary shift, not proof of better reasoning — the optimized module was also confusing the two largest categories in both directions, consistent with pattern-matching a small set of memorized label strings rather than reasoning freshly about each complaint.

Building a proper semantic-equivalence metric (keyword sets per category, not just generic-word exclusion) and scaling the eval set from 10 to 25 examples got me to a result I actually trust: **baseline 0.12, optimized 0.36, a +0.24 gap.** At n=25 this is a real difference, not one example flipping. Manually reviewing the baseline's 22 failures showed the gap is mostly genuine: 45% of baseline failures predicted "Identity Theft" or "Fraud" for complaints that weren't about either, a systematic bias I didn't expect going in — likely the model leaning on fraud-related framing that shows up often among the retrieved similar cases, even when the true issue is more mundane. The optimized module does share less of that specific bias at the n=25 scale — 24% of its failures are identity-theft mispredictions, versus the baseline's 45% — but it trades that bias for a narrower one: 53% of its failures are the optimized module flip-flopping between the two largest categories, "Incorrect information on your report" and "Improper use of your report," in both directions.

**A caveat that changes how I'd frame the +0.24 improvement:** 87% of training examples passed the original `triage_metric` before optimization even ran, which means the optimizer's demonstrations weren't meaningfully curated for correctness — they were drawn from a pool where almost anything using overlapping vocabulary passed. Combined with the failure-pattern shift above, within this evaluation, the observed improvement appears to stem more from adopting CFPB's label vocabulary and defaulting toward the two most frequent categories under uncertainty than from demonstrably better category reasoning. That still produced a real accuracy gain and a real reduction in one specific bias (identity-theft over-prediction) — but it's a narrower, more mechanical effect than "the optimizer improved the model's understanding" would suggest.

**What I'd tell someone using this module:** it works, and optimization measurably helps, but 36% semantic-metric accuracy on issue-category prediction isn't something I'd deploy without a human in the loop. The value this module adds isn't a confident automated categorization — it's a fast, retrieval-grounded starting point (predicted category, suggested approach, confidence level) that a caseworker reviews and corrects, not one that replaces their judgment.

**How this closes out the project:** Notebook 04 established which signals predict `Company response to consumer` and how far structured features versus narrative text each get you. Notebook 05 built and validated a retriever that reliably surfaces similar historical complaints (0.86 mean issue-match rate). This notebook turns that retrieval into an actual triage tool, and in the process surfaced a lesson that applies to the whole pipeline, not just this stage: an evaluation metric that looks reasonable on paper can quietly measure the wrong thing, and the only way to catch that is to actually read the failures instead of trusting the aggregate number.